<img src="https://i.postimg.cc/J7sf3Lq4/Group.png" alt="FUTURA" width="220" />

# Taller 2 — Web Scraping con **Selenium**

En este taller vamos a:

1. **Configurar** un navegador automatizado (Chrome) controlado por Python.
2. **Inspeccionar** una página web (mi CV personal) y extraer datos específicos.
3. **Recorrer** múltiples páginas (Quotes to Scrape) navegando link por link.
4. **Almacenar** los resultados en un `DataFrame` de pandas para análisis.

> 💡 **Concepto clave:** Selenium **maneja un navegador real** (a diferencia de `requests`, que solo descarga HTML). Eso lo hace más lento, pero permite scrapear sitios con JavaScript, formularios, login, scroll infinito, etc.

## Objetivo
Extraer citas, autores, etiquetas y enlaces desde la página Quotes to Scrape, y luego obtener información adicional de cada autor.

In [68]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
import csv
import pandas as pd

In [69]:
#SI NO QUIERES USAR CHROME USA ESTA VAINA XDDD.

In [70]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager # Sigue usando el instalador de chrome, pero forzaremos Brave
import os

# --- 1. CONFIGURAR LA RUTA DE BRAVE ---
# Ruta típica en Windows. Si no funciona, búscala en tu PC y actualiza esta línea.
brave_path = r"C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe"

# Verifica que el archivo exista antes de ejecutar
if not os.path.exists(brave_path):
    print(f"ERROR: No se encontró Brave en: {brave_path}")
    print("Por favor, busca la ruta correcta de 'brave.exe' en tu PC y actualiza el código.")
    # Opción alternativa: Busca en Program Files (x86)
    brave_path = r"C:\Program Files (x86)\BraveSoftware\Brave-Browser\Application\brave.exe"
    if not os.path.exists(brave_path):
        raise FileNotFoundError("No se encontró el ejecutable de Brave en las rutas estándar.")

# --- 2. CONFIGURAR OPCIONES ---
options = Options()
options.binary_location = brave_path  # <--- CLAVE: Le dice a Selenium que use Brave
options.add_argument('--start-maximized')
options.add_argument('--disable-notifications')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage') # Recomendado para contenedores/Linux, pero útil también en Windows

prefs = {"profile.default_content_setting_values.notifications": 2}
options.add_experimental_option("prefs", prefs)

# --- 3. INICIALIZAR EL DRIVER ---
# ChromeDriverManager suele funcionar para Brave si la versión es compatible, 
# pero a veces es mejor descargar el driver de Brave manualmente si falla.
try:
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), 
        options=options
    )
    print("¡Brave se abrió correctamente!")
    
    # Tu código de prueba
    driver.get("https://www.google.com")
    
except Exception as e:
    print(f"Error al iniciar: {e}")
    print("Intenta descargar manualmente el 'BraveDriver' desde la página oficial de Brave y usa esa ruta en Service.")
    # Ejemplo si descargas el driver manualmente:
    # driver = webdriver.Chrome(service=Service(r"C:\ruta\al\bravedriver.exe"), options=options)

¡Brave se abrió correctamente!


In [73]:
driver.quit()

## Configuración del navegador

In [4]:
#SI TIENES CHROME USA ESTO:

In [74]:
options = webdriver.ChromeOptions()
options.add_argument('--start-maximized')
options.add_argument('--disable-notifications')
options.add_argument('--no-sandbox')

prefs = {"profile.default_content_setting_values.notifications": 2}
options.add_experimental_option("prefs", prefs)

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

## Extracción de citas

In [ ]:
url = 'https://quotes.toscrape.com/'
driver.get(url)

In [76]:
main_xpath = '//div[@class = "quote"]'

In [77]:
results = driver.find_elements(By.XPATH, main_xpath)

In [78]:
result = results[0]

In [79]:
result

<selenium.webdriver.remote.webelement.WebElement (session="bc82f51fe63255ab160dba950dbcf26a", element="f.9632FA22D42573DAEC2A4E9043138331.d.5FBFE226BE8E207BA10662A442157DCF.e.5")>

In [80]:
autor_xpath = './span[@class = "text"]/following-sibling::span[1]/small'

In [81]:
autor = result.find_elements(By.XPATH, autor_xpath)[0].text

In [82]:
link_xpath = './span[@class = "text"]/following-sibling::span[1]/a'

In [83]:
link = result.find_elements(By.XPATH, link_xpath)[0].get_attribute('href')

In [84]:
cita_xpath = './span[@class = "text"]'

In [85]:
cita = result.find_elements(By.XPATH, cita_xpath)[0].text

In [86]:
tags_xpath = './div/a'

In [87]:
lista_tags = [i.text for i in result.find_elements(By.XPATH, tags_xpath)]

In [88]:
lista_tags

['change', 'deep-thoughts', 'thinking', 'world']

In [89]:
#Mejor separarlo por '|'
tags = '|'.join(lista_tags)
tags

'change|deep-thoughts|thinking|world'

In [90]:
tags.split()

['change|deep-thoughts|thinking|world']

In [91]:
list_dic = list()
for result in results:
    dict_data=dict()
    ## XPATHS
    autor_xpath = './span[@class = "text"]/following-sibling::span[1]/small'
    link_xpath = './span[@class = "text"]/following-sibling::span[1]/a'
    cita_xpath = './span[@class = "text"]'
    tags_xpath = './div/a'

    #DATOS A EXTRAER:
    dict_data['autor'] = result.find_elements(By.XPATH, autor_xpath)[0].text
    dict_data['link'] = result.find_elements(By.XPATH, link_xpath)[0].get_attribute('href')
    dict_data['cita'] = result.find_elements(By.XPATH, cita_xpath)[0].text
    lista_tags = [i.text for i in result.find_elements(By.XPATH, tags_xpath)]
    dict_data['tags'] = '|'.join(lista_tags)

    list_dic.append(dict_data)

list_dic


[{'autor': 'Albert Einstein',
  'link': 'https://quotes.toscrape.com/author/Albert-Einstein',
  'cita': '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”',
  'tags': 'change|deep-thoughts|thinking|world'},
 {'autor': 'J.K. Rowling',
  'link': 'https://quotes.toscrape.com/author/J-K-Rowling',
  'cita': '“It is our choices, Harry, that show what we truly are, far more than our abilities.”',
  'tags': 'abilities|choices'},
 {'autor': 'Albert Einstein',
  'link': 'https://quotes.toscrape.com/author/Albert-Einstein',
  'cita': '“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”',
  'tags': 'inspirational|life|live|miracle|miracles'},
 {'autor': 'Jane Austen',
  'link': 'https://quotes.toscrape.com/author/Jane-Austen',
  'cita': '“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”',
  'tags': '

In [92]:
data = pd.DataFrame(list_dic)
data

,autor,link,cita,tags
0,Albert Einstein,https://quotes.toscrape.com/author/Albert-Eins...,“The world as we have created it is a process ...,change|deep-thoughts|thinking|world
1,J.K. Rowling,https://quotes.toscrape.com/author/J-K-Rowling,"“It is our choices, Harry, that show what we t...",abilities|choices
2,Albert Einstein,https://quotes.toscrape.com/author/Albert-Eins...,“There are only two ways to live your life. On...,inspirational|life|live|miracle|miracles
3,Jane Austen,https://quotes.toscrape.com/author/Jane-Austen,"“The person, be it gentleman or lady, who has ...",aliteracy|books|classic|humor
4,Marilyn Monroe,https://quotes.toscrape.com/author/Marilyn-Monroe,"“Imperfection is beauty, madness is genius and...",be-yourself|inspirational
5,Albert Einstein,https://quotes.toscrape.com/author/Albert-Eins...,“Try not to become a man of success. Rather be...,adulthood|success|value
6,André Gide,https://quotes.toscrape.com/author/Andre-Gide,“It is better to be hated for what you are tha...,life|love
7,Thomas A. Edison,https://quotes.toscrape.com/author/Thomas-A-Ed...,"“I have not failed. I've just found 10,000 way...",edison|failure|inspirational|paraphrased
8,Eleanor Roosevelt,https://quotes.toscrape.com/author/Eleanor-Roo...,“A woman is like a tea bag; you never know how...,misattributed-eleanor-roosevelt
9,Steve Martin,https://quotes.toscrape.com/author/Steve-Martin,"“A day without sunshine is like, you know, nig...",humor|obvious|simile


In [93]:
list_dic[0].keys()

dict_keys(['autor', 'link', 'cita', 'tags'])

In [94]:
#GUARDAR LOS DATOS
import csv

fuente = 'quote'
path_name = f'./{fuente}.csv'
cabecera = list_dic[0].keys()
with open(path_name, 'w', newline = '') as f:
    writer = csv.DictWriter(
        f,
        delimiter = ',',
        quotechar = '"',
        fieldnames = cabecera
    )
    writer.writeheader()
    writer.writerows(list_dic)

## Lectura del archivo principal

In [24]:
import pandas as pd

In [26]:
page_main = pd.read_csv('quote.csv', encoding='latin_1')
page_main.head()

,autor,link,cita,tags
0,Albert Einstein,https://quotes.toscrape.com/author/Albert-Eins...,The world as we have created it is a process ...,change|deep-thoughts|thinking|world
1,J.K. Rowling,https://quotes.toscrape.com/author/J-K-Rowling,"It is our choices, Harry, that show what we t...",abilities|choices
2,Albert Einstein,https://quotes.toscrape.com/author/Albert-Eins...,There are only two ways to live your life. On...,inspirational|life|live|miracle|miracles
3,Jane Austen,https://quotes.toscrape.com/author/Jane-Austen,"The person, be it gentleman or lady, who has ...",aliteracy|books|classic|humor
4,Marilyn Monroe,https://quotes.toscrape.com/author/Marilyn-Monroe,"Imperfection is beauty, madness is genius and...",be-yourself|inspirational


In [27]:
links=page_main.link.to_list()

In [28]:
link=links[0]

In [29]:
driver.get(link)

In [30]:
main_xpath = "//div[@class='author-details']/h3/following-sibling::p[1]"

In [42]:
results = driver.find_elements(By.XPATH, main_xpath)[0]
results

<selenium.webdriver.remote.webelement.WebElement (session="d6dd7ecaac3ea2710d0f19ea9ddffcec", element="f.91DF9CA32F7D19EAD2EA2CFE36EFCA1F.d.74B55C85A1CEEBFE41062E8D2922DA35.e.3")>

In [32]:
date_xpath ="./span[contains(@class,'date')]"

In [44]:
date = results.find_elements(By.XPATH, date_xpath)[0].text

In [45]:
date

'March 14, 1879'

In [46]:
location_xpath = "./span[contains(@class,'location')]"

In [48]:
location = results.find_elements(By.XPATH, location_xpath)[0].text
location

'in Ulm, Germany'

## Extracción de información de autores

In [52]:
list_data = list()

for link in links:
        driver.get(link)
        dict_data = dict()
    
        #PATHS
        main_xpath = "//div[@class='author-details']/h3/following-sibling::p[1]"
        date_xpath ="./span[contains(@class,'date')]"
        location_xpath ="./span[contains(@class,'location')]"
        link_xpath = './span[@class = "text"]/following-sibling::span[1]/a'

    
        #Extraer datos
        results = driver.find_elements(By.XPATH, main_xpath)[0]
        dict_data['date'] = results.find_elements(By.XPATH, date_xpath)[0].text
        dict_data['location'] = results.find_elements(By.XPATH, location_xpath)[0].text
        dict_data['link'] = link

        list_data.append(dict_data)

list_data

[{'date': 'March 14, 1879',
  'location': 'in Ulm, Germany',
  'link': 'https://quotes.toscrape.com/author/Albert-Einstein'},
 {'date': 'July 31, 1965',
  'location': 'in Yate, South Gloucestershire, England, The United Kingdom',
  'link': 'https://quotes.toscrape.com/author/J-K-Rowling'},
 {'date': 'March 14, 1879',
  'location': 'in Ulm, Germany',
  'link': 'https://quotes.toscrape.com/author/Albert-Einstein'},
 {'date': 'December 16, 1775',
  'location': 'in Steventon Rectory, Hampshire, The United Kingdom',
  'link': 'https://quotes.toscrape.com/author/Jane-Austen'},
 {'date': 'June 01, 1926',
  'location': 'in The United States',
  'link': 'https://quotes.toscrape.com/author/Marilyn-Monroe'},
 {'date': 'March 14, 1879',
  'location': 'in Ulm, Germany',
  'link': 'https://quotes.toscrape.com/author/Albert-Einstein'},
 {'date': 'November 22, 1869',
  'location': 'in Paris, France',
  'link': 'https://quotes.toscrape.com/author/Andre-Gide'},
 {'date': 'February 11, 1847',
  'location

In [53]:
len(list_data)

10

In [54]:
list_data[0].keys()

dict_keys(['date', 'location', 'link'])

In [62]:
#GUARDAR LOS DATOS
import csv

fuente = 'quote_info_autor'
path_name = f'./{fuente}.csv'
cabecera = list_data[0].keys()
with open(path_name, 'w', newline = '') as f:
    writer = csv.DictWriter(
        f,
        delimiter = ',',
        quotechar = '"',
        fieldnames = cabecera
    )
    writer.writeheader()
    writer.writerows(list_data)

## Cierre del navegador

In [67]:
driver.quit()

In [96]:
tabla1 = pd.read_csv('quote_info_autor.csv' , encoding = 'latin1')
tabla1

,date,location,link
0,"March 14, 1879","in Ulm, Germany",https://quotes.toscrape.com/author/Albert-Eins...
1,"July 31, 1965","in Yate, South Gloucestershire, England, The U...",https://quotes.toscrape.com/author/J-K-Rowling
2,"March 14, 1879","in Ulm, Germany",https://quotes.toscrape.com/author/Albert-Eins...
3,"December 16, 1775","in Steventon Rectory, Hampshire, The United Ki...",https://quotes.toscrape.com/author/Jane-Austen
4,"June 01, 1926",in The United States,https://quotes.toscrape.com/author/Marilyn-Monroe
5,"March 14, 1879","in Ulm, Germany",https://quotes.toscrape.com/author/Albert-Eins...
6,"November 22, 1869","in Paris, France",https://quotes.toscrape.com/author/Andre-Gide
7,"February 11, 1847","in Milan, Ohio, The United States",https://quotes.toscrape.com/author/Thomas-A-Ed...
8,"October 11, 1884",in The United States,https://quotes.toscrape.com/author/Eleanor-Roo...
9,"August 14, 1945","in Waco, Texas, The United States",https://quotes.toscrape.com/author/Steve-Martin


In [97]:
tabla2= pd.read_csv('quote.csv' , encoding = 'latin1')
tabla2

,autor,link,cita,tags
0,Albert Einstein,https://quotes.toscrape.com/author/Albert-Eins...,The world as we have created it is a process ...,change|deep-thoughts|thinking|world
1,J.K. Rowling,https://quotes.toscrape.com/author/J-K-Rowling,"It is our choices, Harry, that show what we t...",abilities|choices
2,Albert Einstein,https://quotes.toscrape.com/author/Albert-Eins...,There are only two ways to live your life. On...,inspirational|life|live|miracle|miracles
3,Jane Austen,https://quotes.toscrape.com/author/Jane-Austen,"The person, be it gentleman or lady, who has ...",aliteracy|books|classic|humor
4,Marilyn Monroe,https://quotes.toscrape.com/author/Marilyn-Monroe,"Imperfection is beauty, madness is genius and...",be-yourself|inspirational
5,Albert Einstein,https://quotes.toscrape.com/author/Albert-Eins...,Try not to become a man of success. Rather be...,adulthood|success|value
6,André Gide,https://quotes.toscrape.com/author/Andre-Gide,It is better to be hated for what you are tha...,life|love
7,Thomas A. Edison,https://quotes.toscrape.com/author/Thomas-A-Ed...,"I have not failed. I've just found 10,000 way...",edison|failure|inspirational|paraphrased
8,Eleanor Roosevelt,https://quotes.toscrape.com/author/Eleanor-Roo...,A woman is like a tea bag; you never know how...,misattributed-eleanor-roosevelt
9,Steve Martin,https://quotes.toscrape.com/author/Steve-Martin,"A day without sunshine is like, you know, nig...",humor|obvious|simile


In [106]:
table = tabla1.merge(tabla2 , on = 'link', how = 'left').drop_duplicates()
table

,date,location,link,autor,cita,tags
0,"March 14, 1879","in Ulm, Germany",https://quotes.toscrape.com/author/Albert-Eins...,Albert Einstein,The world as we have created it is a process ...,change|deep-thoughts|thinking|world
1,"March 14, 1879","in Ulm, Germany",https://quotes.toscrape.com/author/Albert-Eins...,Albert Einstein,There are only two ways to live your life. On...,inspirational|life|live|miracle|miracles
2,"March 14, 1879","in Ulm, Germany",https://quotes.toscrape.com/author/Albert-Eins...,Albert Einstein,Try not to become a man of success. Rather be...,adulthood|success|value
3,"July 31, 1965","in Yate, South Gloucestershire, England, The U...",https://quotes.toscrape.com/author/J-K-Rowling,J.K. Rowling,"It is our choices, Harry, that show what we t...",abilities|choices
7,"December 16, 1775","in Steventon Rectory, Hampshire, The United Ki...",https://quotes.toscrape.com/author/Jane-Austen,Jane Austen,"The person, be it gentleman or lady, who has ...",aliteracy|books|classic|humor
8,"June 01, 1926",in The United States,https://quotes.toscrape.com/author/Marilyn-Monroe,Marilyn Monroe,"Imperfection is beauty, madness is genius and...",be-yourself|inspirational
12,"November 22, 1869","in Paris, France",https://quotes.toscrape.com/author/Andre-Gide,André Gide,It is better to be hated for what you are tha...,life|love
13,"February 11, 1847","in Milan, Ohio, The United States",https://quotes.toscrape.com/author/Thomas-A-Ed...,Thomas A. Edison,"I have not failed. I've just found 10,000 way...",edison|failure|inspirational|paraphrased
14,"October 11, 1884",in The United States,https://quotes.toscrape.com/author/Eleanor-Roo...,Eleanor Roosevelt,A woman is like a tea bag; you never know how...,misattributed-eleanor-roosevelt
15,"August 14, 1945","in Waco, Texas, The United States",https://quotes.toscrape.com/author/Steve-Martin,Steve Martin,"A day without sunshine is like, you know, nig...",humor|obvious|simile


In [107]:
driver.quit()